In [2]:
import sys
sys.path.append('/opt/homebrew/Cellar/apache-spark/3.5.5/libexec/python')
sys.path.append('/opt/homebrew/Cellar/apache-spark/3.5.5/libexec/python/lib/py4j-0.10.9.7-src.zip')

In [3]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/09 13:25:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 56627)
Traceback (most recent call last):
  File "/Users/nikolai/anaconda3/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/nikolai/anaconda3/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/Users/nikolai/anaconda3/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/nikolai/anaconda3/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/opt/homebre

In [5]:
# !wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [6]:
# !gzip -dc fhvhv_tripdata_2021-01.csv.gz

In [7]:
!wc -l fhvhv_tripdata_2021-01.csv

 11908469 fhvhv_tripdata_2021-01.csv


In [9]:
%%time
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

CPU times: user 1.42 ms, sys: 2.05 ms, total: 3.47 ms
Wall time: 132 ms


In [10]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [11]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [14]:
import pandas as pd

In [15]:
df_pandas = pd.read_csv('head.csv')

In [16]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [17]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [18]:
from pyspark.sql import types

In [19]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [20]:
%%time
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

CPU times: user 2.87 ms, sys: 3.19 ms, total: 6.07 ms
Wall time: 29.3 ms


In [21]:
df = df.repartition(24)

In [23]:
%%time
df.write.parquet('fhvhv/2021/01/')

25/03/09 13:29:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/03/09 13:29:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/03/09 13:29:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


CPU times: user 7.6 ms, sys: 4.84 ms, total: 12.4 ms
Wall time: 8.84 s


In [24]:
df = spark.read.parquet('fhvhv/2021/01/')

In [25]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [26]:
from pyspark.sql import functions as F

In [27]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-01 16:08:13|2021-01-01 16:11:57|          37|          36|   NULL|
|           HV0005|              B02510|2021-01-04 09:14:45|2021-01-04 09:30:04|          89|         165|   NULL|
|           HV0003|              B02617|2021-01-05 10:50:11|2021-01-05 11:05:40|         161|         238|   NULL|
|           HV0003|              B02617|2021-01-03 10:21:50|2021-01-03 10:50:41|         225|         231|   NULL|
|           HV0005|              B02510|2021-01-01 03:53:08|2021-01-01 04:02:54|         243|         244|   NULL|
|           HV0003|              B02877|2021-01-01 02:02:05|2021-01-01 02:10:12|

In [28]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [29]:
crazy_stuff('B02884')

's/b44'

In [30]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [31]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-01|  2021-01-01|          37|          36|
|  e/9ce| 2021-01-04|  2021-01-04|          89|         165|
|  e/a39| 2021-01-05|  2021-01-05|         161|         238|
|  e/a39| 2021-01-03|  2021-01-03|         225|         231|
|  e/9ce| 2021-01-01|  2021-01-01|         243|         244|
|  s/b3d| 2021-01-01|  2021-01-01|         202|         146|
|  e/b30| 2021-01-04|  2021-01-04|         254|          81|
|  a/b43| 2021-01-02|  2021-01-02|          61|          61|
|  s/b44| 2021-01-03|  2021-01-03|         107|         170|
|  e/b32| 2021-01-02|  2021-01-02|          29|         210|
|  e/acc| 2021-01-01|  2021-01-01|         231|         144|
|  e/acc| 2021-01-02|  2021-01-02|         138|         263|
|  e/b3e| 2021-01-04|  2021-01-04|         237|         163|
|  a/b40| 2021-01-03|  2

In [32]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [34]:
# !head -n 10 head.csv

In [35]:
spark.stop()

ConnectionRefusedError: [Errno 61] Connection refused